In [3]:
#packages
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


In [4]:
#files and configuration
MASTER_PATH = r"D:\machine learning for finance literature\cleaned x variables (final one)\master_panel_v5.csv"
DV_PATH = r"D:\machine learning for finance literature\cleaned x variables (final one)\portfolio_investment_final.csv"

PREDICTOR_COLS = [
    "exchange_rate", "reserves_pct_gdp", "gdp_growth_yoy", "m2_growth_yoy",
    "market_cap_gdp_ratio", "Interest_Rate", "Stock_Return_Percent",
    "Stock_Volatility", "us_dollar_index", "VIX", "Fed_Funds_Rate",
    "Brent_Oil_Price", "current_account_pct_gdp",
]
TRAIN_FRACTION = 0.80
WALK_FORWARD_START = "2010Q1"   # first quarter to be predicted out-of-sample


def flag_reversals_percentile(series, pct=0.10, min_periods=12):
    flags = pd.Series(index=series.index, dtype=float)
    for i in range(len(series)):
        if i + 1 < min_periods:
            flags.iloc[i] = np.nan
        else:
            threshold = series.iloc[: i + 1].quantile(pct)
            flags.iloc[i] = int(series.iloc[i] <= threshold)
    return flags

In [5]:
#loading and cleaning
X_df = pd.read_csv(MASTER_PATH)
dv_df = pd.read_csv(DV_PATH)
dv_df["portfolio_investment"] = pd.to_numeric(
    dv_df["portfolio_investment"].astype(str).str.replace(",", "", regex=False), errors="coerce"
)
dv_df = dv_df.dropna(subset=["portfolio_investment"])
dv_df = dv_df.sort_values(["country", "quarter"])
dv_df["reversal"] = dv_df.groupby("country")["portfolio_investment"].transform(
    lambda s: flag_reversals_percentile(s.reset_index(drop=True), pct=0.10).values
)

for col in PREDICTOR_COLS:
    if col in X_df.columns:
        X_df[col] = pd.to_numeric(
            X_df[col].astype(str).str.replace(",", "", regex=False), errors="coerce"
        )
X_df = X_df.sort_values(["country", "quarter"])
for col in PREDICTOR_COLS:
    X_df[col] = X_df.groupby("country")[col].shift(1)

panel = X_df.merge(dv_df[["country", "quarter", "reversal"]], on=["country", "quarter"], how="inner")
use = panel.dropna(subset=PREDICTOR_COLS + ["reversal"]).reset_index(drop=True)

use["quarter_dt"] = pd.PeriodIndex(use["quarter"], freq="Q").to_timestamp()
use = use.sort_values("quarter_dt").reset_index(drop=True)

print("Observations:", len(use))
print("Event rate:", round(use["reversal"].mean(), 3))


Observations: 1555
Event rate: 0.166


In [6]:
#80/20 split
unique_quarters = use["quarter_dt"].sort_values().unique()
cutoff_index = int(len(unique_quarters) * TRAIN_FRACTION)
cutoff_date = unique_quarters[cutoff_index]

train = use[use["quarter_dt"] < cutoff_date].reset_index(drop=True)
test = use[use["quarter_dt"] >= cutoff_date].reset_index(drop=True)

print(f"Cutoff date: {pd.Timestamp(cutoff_date).date()}")
print(f"Train: {len(train)} obs, {train['quarter_dt'].min().date()} to {train['quarter_dt'].max().date()}, "
      f"event rate={train['reversal'].mean():.3f}")
print(f"Test:  {len(test)} obs, {test['quarter_dt'].min().date()} to {test['quarter_dt'].max().date()}, "
      f"event rate={test['reversal'].mean():.3f}")

X_train_raw = train[PREDICTOR_COLS].values.astype(float)
y_train = train["reversal"].values.astype(int)
X_test_raw = test[PREDICTOR_COLS].values.astype(float)
y_test = test["reversal"].values.astype(int)

Cutoff date: 2021-04-01
Train: 1234 obs, 2002-10-01 to 2021-01-01, event rate=0.159
Test:  321 obs, 2021-04-01 to 2025-10-01, event rate=0.193


In [7]:
#fitting logit single split
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_raw)
X_test_s = scaler.transform(X_test_raw)

model = LogisticRegression(C=1e10, max_iter=5000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]   # predicted P(reversal)

print("\n=== OUT-OF-SAMPLE TEST RESULTS (80/20 time split) ===")
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test F1:        {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Test AUROC:     {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nNaive baseline (always predict 'no reversal') on test set: {1 - y_test.mean():.4f}")



=== OUT-OF-SAMPLE TEST RESULTS (80/20 time split) ===
Test accuracy:  0.7227
Test precision: 0.1538
Test recall:    0.0968
Test F1:        0.1188
Test AUROC:     0.4070

Naive baseline (always predict 'no reversal') on test set: 0.8069


In [8]:
#coef and confussion matrix
coef_original = model.coef_[0] / scaler.scale_
coef_table = pd.Series(coef_original, index=PREDICTOR_COLS)
print("\n=== Coefficients (original scale) ===")
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 100)
print(coef_table)

cm_table = pd.crosstab(
    pd.Series(y_pred, name="Predicted"),
    pd.Series(y_test, name="True reversal status"),
    margins=True, margins_name="Total"
)
cm_table.index = ["No Reversal", "Reversal", "Total"]
cm_table.columns = ["No Reversal", "Reversal", "Total"]
print("\n=== Confusion Matrix ===")
print(cm_table.to_string())


=== Coefficients (original scale) ===
exchange_rate             -0.000028
reserves_pct_gdp           0.019575
gdp_growth_yoy            -0.386510
m2_growth_yoy             -0.181437
market_cap_gdp_ratio      -0.000222
Interest_Rate             -0.005754
Stock_Return_Percent      -0.010777
Stock_Volatility           0.037915
us_dollar_index            0.040898
VIX                       -0.003828
Fed_Funds_Rate             0.150506
Brent_Oil_Price            0.015322
current_account_pct_gdp   -0.012495
dtype: float64

=== Confusion Matrix ===
             No Reversal  Reversal  Total
No Reversal          226        56    282
Reversal              33         6     39
Total                259        62    321


In [9]:
#walk forward
wf_start = pd.Period(WALK_FORWARD_START, freq="Q").to_timestamp()
eval_quarters = [q for q in unique_quarters if q >= wf_start]

rows = []
all_true, all_prob, all_pred = [], [], []

for q in eval_quarters:
    tr = use[use["quarter_dt"] < q]
    te = use[use["quarter_dt"] == q]
    if len(te) == 0 or tr["reversal"].nunique() < 2:
        continue  # need both classes in training to fit a logit

    sc = StandardScaler()
    Xtr = sc.fit_transform(tr[PREDICTOR_COLS].values.astype(float))
    Xte = sc.transform(te[PREDICTOR_COLS].values.astype(float))
    ytr = tr["reversal"].values.astype(int)
    yte = te["reversal"].values.astype(int)

    m = LogisticRegression(C=1e10, max_iter=5000)
    m.fit(Xtr, ytr)

    pred = m.predict(Xte)
    prob = m.predict_proba(Xte)[:, 1]

    rows.append({
        "quarter": pd.Timestamp(q).to_period("Q").strftime("%YQ%q"),
        "n_obs": len(te),
        "n_reversals": int(yte.sum()),
        "accuracy": accuracy_score(yte, pred),
    })
    all_true.extend(yte)
    all_prob.extend(prob)
    all_pred.extend(pred)

wf = pd.DataFrame(rows)
all_true = np.array(all_true)
all_prob = np.array(all_prob)
all_pred = np.array(all_pred)

print("Completed quarters:", len(wf), "/", len(eval_quarters))


Completed quarters: 64 / 64


In [10]:
#summary 
print(f"\n=== WALK-FORWARD EVALUATION (from {WALK_FORWARD_START}) ===")
print(f"Quarters evaluated: {len(wf)} "
      f"({wf['quarter'].iloc[0]} to {wf['quarter'].iloc[-1]})")
print(f"Total predictions:  {len(all_true)} (of which {int(all_true.sum())} true reversals)")
print(f"\nAverage prediction accuracy (mean across quarters): {wf['accuracy'].mean():.4f}")
print(f"Pooled accuracy (all obs together):                 {accuracy_score(all_true, all_pred):.4f}")
print(f"Pooled AUROC:                                       {roc_auc_score(all_true, all_prob):.4f}")
print(f"Pooled F1:                                          {f1_score(all_true, all_pred, zero_division=0):.4f}")
print(f"Pooled recall:                                      {recall_score(all_true, all_pred, zero_division=0):.4f}")
print(f"Naive baseline accuracy (always 'no reversal'):     {1 - all_true.mean():.4f}")

print("\n=== Per-quarter accuracy (walk-forward) ===")
print(wf.to_string(index=False))


=== WALK-FORWARD EVALUATION (from 2010Q1) ===
Quarters evaluated: 64 (2010Q1 to 2025Q4)
Total predictions:  1220 (of which 201 true reversals)

Average prediction accuracy (mean across quarters): 0.8243
Pooled accuracy (all obs together):                 0.8180
Pooled AUROC:                                       0.5344
Pooled F1:                                          0.0902
Pooled recall:                                      0.0547
Naive baseline accuracy (always 'no reversal'):     0.8352

=== Per-quarter accuracy (walk-forward) ===
quarter  n_obs  n_reversals  accuracy
 2010Q1     15            2  0.866667
 2010Q2     15            2  0.866667
 2010Q3     15            0  0.933333
 2010Q4     15            1  0.933333
 2011Q1     15            0  1.000000
 2011Q2     17            2  0.882353
 2011Q3     17            4  0.705882
 2011Q4     17            2  0.823529
 2012Q1     17            0  1.000000
 2012Q2     20            3  0.850000
 2012Q3     20            0  0.900000


In [11]:
wf_export = wf.copy()
wf_export["model"] = "Logit"
wf_export.to_csv("walkforward_logit.csv", index=False)
print("\nSaved: walkforward_logit.csv")


Saved: walkforward_logit.csv
